# RNFLT features: MLP vs. XGBoost



In [5]:
import os

from sklearn.model_selection import KFold
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import time


PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "scripts" / "load_data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate scripts/load_data.py from the notebook working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.load_data import load_data

data, feature_sets = load_data()

SEED = 42
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 10
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
train_mask = data["split"].eq("train").to_numpy()
val_mask = data["split"].eq("val").to_numpy()
test_mask = data["split"].eq("test").to_numpy()
y = data["label"].to_numpy(dtype=np.int64)
y_train, y_val, y_test = y[train_mask], y[val_mask], y[test_mask]

class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def forward(self, inputs):
        return self.network(inputs)

def train_model(columns):
    raw = data[columns].to_numpy(dtype=np.float32)
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    x_train = scaler.fit_transform(imputer.fit_transform(raw[train_mask])).astype(np.float32)
    x_val = scaler.transform(imputer.transform(raw[val_mask])).astype(np.float32)
    x_test = scaler.transform(imputer.transform(raw[test_mask])).astype(np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    torch.manual_seed(SEED)
    model = MultiLayerPerceptron(input_size=len(columns)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    x_val_tensor = torch.from_numpy(x_val).to(DEVICE)
    y_val_tensor = torch.from_numpy(y_val).to(DEVICE)
    best_loss, best_state, best_epoch, stale_epochs = float("inf"), None, 0, 0
    start_time = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val_tensor), y_val_tensor).item()
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                break
    if best_state is None:
        raise RuntimeError("Training did not produce a valid model checkpoint.")
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_prob = torch.softmax(model(x_val_tensor), dim=1)[:, 1].cpu().numpy()
        test_prob = torch.softmax(model(torch.from_numpy(x_test).to(DEVICE)), dim=1)[:, 1].cpu().numpy()
    end_time = time.perf_counter()
    print(f"MLP training time: {end_time - start_time:.2f} seconds")
    return val_prob, test_prob, best_epoch, best_loss

def train_xgboost(columns):
    raw = data[columns].to_numpy(dtype=np.float32)
    imputer = SimpleImputer(strategy="median")
    x_train = imputer.fit_transform(raw[train_mask]).astype(np.float32)
    x_val = imputer.transform(raw[val_mask]).astype(np.float32)
    x_test = imputer.transform(raw[test_mask]).astype(np.float32)
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.9,
        reg_lambda=2.0,
        early_stopping_rounds=30,
        n_jobs=1,
        random_state=SEED,
        tree_method="hist",
    )
    start_time = time.perf_counter()
    model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    end_time = time.perf_counter()
    print(f"XGBoost training time: {end_time - start_time:.2f} seconds")
    return model.predict_proba(x_val)[:, 1], model.predict_proba(x_test)[:, 1], model.best_iteration

def expected_calibration_error(y_true, probabilities, n_bins=10):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for index in range(n_bins):
        in_bin = (probabilities >= edges[index]) & (probabilities <= edges[index + 1]) if index == 0 else (
            (probabilities > edges[index]) & (probabilities <= edges[index + 1])
        )
        if in_bin.any():
            correct = np.mean((probabilities[in_bin] >= 0.5) == y_true[in_bin])
            confidence = np.mean(np.maximum(probabilities[in_bin], 1 - probabilities[in_bin]))
            ece += in_bin.mean() * abs(correct - confidence)
    return float(ece)

def bootstrap_auc_ci(y_true, probabilities, n_boot=2000, seed=SEED):
    """Return a stratified percentile 95% CI for AUROC on fixed predictions."""
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)
    if y_true.ndim != 1 or y_true.shape != probabilities.shape:
        raise ValueError("Labels and probabilities must be same-length 1D arrays.")
    if n_boot < 1:
        raise ValueError("n_boot must be at least 1.")
    positive = np.flatnonzero(y_true == 1)
    negative = np.flatnonzero(y_true == 0)
    if positive.size == 0 or negative.size == 0 or positive.size + negative.size != y_true.size:
        raise ValueError("Bootstrap AUROC requires both classes and binary labels encoded as 0 and 1.")

    rng = np.random.default_rng(seed)
    bootstrap_aucs = np.empty(n_boot, dtype=float)
    for iteration in range(n_boot):
        sampled_indices = np.concatenate((
            rng.choice(positive, size=positive.size, replace=True),
            rng.choice(negative, size=negative.size, replace=True),
        ))
        bootstrap_aucs[iteration] = roc_auc_score(
            y_true[sampled_indices], probabilities[sampled_indices]
        )
    return np.percentile(bootstrap_aucs, [2.5, 97.5])

k_folds = 5
batch_size = 32
N_BOOTSTRAP = 2000

rows = []
for model_name, columns in feature_sets.items():
    mlp_val_prob, mlp_test_prob, best_epoch, best_loss = train_model(columns)
    #xgb_val_prob, xgb_test_prob, best_iteration = train_xgboost(columns)
    #print(f"{model_name}: MLP best epoch={best_epoch}; XGBoost best iteration={best_iteration}")
    print(f"{model_name}: MLP best epoch={best_epoch}}}")

    for algorithm, val_prob, test_prob in (
        ("MLP", mlp_val_prob, mlp_test_prob),
        #("XGBoost", xgb_val_prob, xgb_test_prob),
    ):
        for split_name, split_mask, split_labels, probabilities in (
            ("Validation", val_mask, y_val, val_prob),
            ("Test", test_mask, y_test, test_prob),
        ):
            race_values = data.loc[split_mask, "race"].to_numpy()
            for race in ("white", "black", "asian", "overall"):
                race_mask = np.ones(len(split_labels), dtype=bool) if race == "overall" else race_values == race
                race_labels = split_labels[race_mask]
                race_prob = probabilities[race_mask]
                if np.unique(race_labels).size != 2:
                    raise ValueError(f"{split_name}/{race} subset must contain both label classes.")
                auc_ci_lower, auc_ci_upper = bootstrap_auc_ci(
                    race_labels, race_prob, n_boot=N_BOOTSTRAP, seed=SEED
                )
                rows.append({
                    "model": f"{algorithm} | {model_name}",
                    "split": split_name,
                    "race": race.title(),
                    "AUROC": roc_auc_score(race_labels, race_prob),
                    "AUROC_CI_95_Lower": auc_ci_lower,
                    "AUROC_CI_95_Upper": auc_ci_upper,
                    "ECE_10_bins": expected_calibration_error(race_labels, race_prob),
                    "n": int(race_mask.sum()),
                    "glaucoma_cases": int(race_labels.sum()),
  
                })

results_df = pd.DataFrame(rows)
print(results_df)
results_df.to_csv(str(PROJECT_ROOT) + '/mlp_xgboost_results.csv', index=False)



Data: /Users/freddy/projects/Glaucoma-Classification-Project/data/processed | rows by split: {'train': 2100, 'test': 900, 'val': 300}
RNFLT predictors: ['hcdr', 'rim_px', 'rnfl_mean', 'rnfl_p10', 'ring_mean', 'ring_sup', 'ring_inf', 'ring_horiz', 'si_asym']
Missing feature values: 11
MLP training time: 4.44 seconds
RNFLT + age: MLP best epoch=39}
MLP training time: 3.37 seconds
RNFLT only (no age): MLP best epoch=27}
                        model       split     race     AUROC  \
0           MLP | RNFLT + age  Validation    White  0.834334   
1           MLP | RNFLT + age  Validation    Black  0.808322   
2           MLP | RNFLT + age  Validation    Asian  0.863231   
3           MLP | RNFLT + age  Validation  Overall  0.824368   
4           MLP | RNFLT + age        Test    White  0.882892   
5           MLP | RNFLT + age        Test    Black  0.793254   
6           MLP | RNFLT + age        Test    Asian  0.874911   
7           MLP | RNFLT + age        Test  Overall  0.849984   
8  